In [ ]:
day08

0. 복습
함수 매핑
 : 데이터 프레임의 각 원소에 함수를 적용하여 값을 변환하는 방법
1) map() : 시리즈(열)의 각 원소를 다른 값으로 변환 
	- 딕셔너리 매핑 : 특정 값을 정해진 다른 값으로 변환
	- 함수 매핑 : lambda(람다)를 적용하여 변환
	ex) male -> '남성', female -> '여성'

2) apply() : 시리즈(열)의 각 원소에 함수를 적용하여 변환

그룹 연산
- 데이터를 어떤 기준에 따라 여러 그룹으로 나눠 연산할 때 사용
ex) 객실등급별 평균 나이 
- df.groupby() : 특정 열을 기준으로 그룹 객체를 반환
ex) df.groupby('class')['age'].mean()
- 그룹객체.agg(매핑함수)
	: 여러개의 함수를 사용하여 각 그룹별로 집계 연산 처리
	그룹객체.agg([함수1,...])
	그룹객체.agg({"열1" : 함수명1,...})

1. 피벗테이블
 : 행과 열 두가지 기준을 동시에 사용하여 데이터를 교차 집계하는 방법
- 엑셀의 피벗테이블과 같은 개념
ex) 객실 등급별/성별 평균 요금이 얼마인가?
- 피벗 테이블 관련 함수
	(1) pivot_table() : 교차 기준으로 데이터를 집계한다
	(2) crosstab() : 두 기준의 조합별 건수(빈도수)를 표 형태로 집계
	(3) pivot() : 집계 없이 데이터의 모양(구조)만 변환한다

1) pivot_table() : 교차 집계표
	옵션		의미
===========================================================
	values		집계할 열(어떤 데이터로 계산할지)
	index		행 기준(표의 세로축)
	columns		열 기준(표의 가로축)
	aggfunc		집계 함수(기본값 : 'mean' 평균)
	fill_value	NaN(결측값)을 대체할 값
	margins		True 설정 시 전체 합계 행/열 추가

2) crosstab() : 빈도수 교차표
- pd.crosstab(index, columns)형태로 사용
- 두 열의 값 조합별로 몇 건(행)이 있는지 건수를 자동으로 집계한다
- normalize = True 옵션을 추가하면 건수 대신 비율(%)로 변환할 수 있다

3) pivot() : 단순 재구조화
- df.pivot(index, columns, values)
- 집계(계산)없이 데이터의 모양(구조)만 바꿀 때 사용
- 주의 : 행/열 기준의 조합에 중복 값이 있으면 오류가 발생한다
  => 중복이 있는 실제 데이터에는 pivot_table()을 사용한다

**피벗테이블**

In [ ]:
# 데이터 불러오기
import pandas as pd
import seaborn as sns

# 타이타닉 데이터셋 불러오기
df = sns.load_dataset("titanic")

df.head()

**1) pivot_table()**

In [ ]:
# 객실 등급별(class), 성별별 평균 요금을 표형태로 생성

# pd.pivot_table() : 교차 집계표를 만드는 함수
# values='fare' : 집계할 열 -> 요급을 계산
# index='class' : 표의 행(세로축) 기준 -> 객실 등급
# columns='sex' : 표의 열(가로축) 기준 -> 성별
# aggfunc='mean' : 집계 방식 -> 평균 계산

result = pd.pivot_table(df, values='fare', index='class', columns='sex', aggfunc='mean')

result.round(2)

In [ ]:
# aggfunc를 바꾸면 다른 통계도 표 형태로 만들 수 있다

# count() : 데이터 개수(건수) => 인원수 집계
result_count = pd.pivot_table(df, values='fare', index='class', columns='sex', aggfunc='count')

result_count

In [ ]:
# margins = True : 표의 마지막에 전체 평균울 나타내는 'All' 행과 열을 자동으로 추가
# fill_value = 0 : 집계할 대상(fare)에 데이터가 없어 NaN이 된 칸을 0으로 채움

result_margins = pd.pivot_table(
    df,
    values='fare',
    index='class',
    columns='sex',
    aggfunc='mean',
    fill_value= 0,
    margins=True
)

result_margins.round(2)

**crosstab()**

In [ ]:
# 객실 등급별(class), 생존 여부(survived)별 승객 수를 표 형태로 집계

# pd.crosstab() : 두 열의 조합별 건수(몇명인지)를 세는 함수
result = pd.crosstab(df['class'], df['survived'])

result

In [ ]:
# normalize=True : 건수 대신에 비율로 변환(각 행의 합계가 1.0이 됨)

# nomalize = True : 전체 기준(전체의 합계가 1.0)
# nomalize = 'index' : 행 기준(각 행의 합계가 1.0)
# nomalize = 'columns' : 열 기준(각 열의 합계가 1.0)

result_ratio = pd.crosstab(df['class'], df['survived'], normalize=True)

result_ratio.round(2)

**pivot()**

In [ ]:
import pandas as pd

# 데이터 프레임 생성
# 요일별, 시간대별 매출 데이터를 표 형태로 재구조화(재구성)
# => pivot()을 사용하여 재구성할때는 요일과 시간대의 조합이 모두 유일(중복 없이)해야 한다
data = {
    "요일" : ['월', '화', '수', '월', '화', '수'],
    '시간대' : ['점심', '점심', '점심', '저녁', '저녁', '저녁'],
    '매출' : [120, 150, 130, 200, 180, 210]
}

sample_df = pd.DataFrame(data)

sample_df

In [ ]:
# pivot() : 집계 없이 데이터의 모양만 변화

result = sample_df.pivot(index='요일', columns='시간대', values='매출')

result

In [ ]:
# 아래처럼 동일한 요일 + 시간대 조합이 2개 이상 있으면 pivot()은 오류 발생!
data = {
    '요일' : ['월', ' 월', '월'], # '월' + '점심' 조합이 2번 등장 => 중복
    '시간대' : ['점심', '점심', '점심'],
    '매출' : [120, 130, 150]
}

df_dup = pd.DataFrame(data)

# df_dup.pivot(index='요일', columns='시간대', values='매출') # => 오류 발생!

# 중복이 있을때는 pivot_table()을 사용해야 한다
result_pt = pd.pivot_table(df_dup, values="매출", index='요일', columns='시간대', aggfunc='mean')
result_pt

In [ ]:
# <피벗 테이블 실습>
import pandas as pd
import seaborn as sns

df = sns.load_dataset("tips")
df.head()

In [ ]:
# 1) pivot_table()을 사용하여,
#   요일(day)별, 흡연 여부(smoker)별 평균 팁(tip)을 표 형태로 출력
#   전체 평균(margins=True)도 함께 확인

# pivot_table() : 요일과 흡연여부 두 기준으로 팁의 평균을 집계
result = pd.pivot_table(
    df,
    values='tip',
    index='day',
    columns='smoker',
    aggfunc='mean',
    margins=True
)

result.round(2)

In [ ]:
# 2) crosstab()을 사용하여,
#   성별(sex)과 흡연 여부(smoker)의 조합별 손님 수를 출력
#   normalize를 적용하여 성별 기준 비율도 함께 확인

result = pd.crosstab(df['sex'], df['smoker'], normalize='index')

result.round(2)

## 종합 분석(타이타닉 생존 분석)
- 타이타닉 사고에서 누가 살아남았는지? 생존에 영향을 미친 요인은 무엇인지?
1. 데이터 탐색
2. 누락 데이터 처리
3. 열 가공(함수 매핑)
4. 조건별 탐색(필터링)
5. 그룹 통계(그룹 연산)
6. 교차 분석(피벗 테이블)
7. 시각화 (그래프 시각화)

In [ ]:
# 나눔 폰트 설치 (한글이 깨지지 않도록 설정)
!sudo apt-get install -y fonts-nanum    # 나눔 폰트 패키지 설치
!sudo fc-cache -fv                      # 폰트 캐시(폰트 목록) 갱신
!rm ~/.cache/matplotlib -rf             # matplotlib 폰트 캐시 삭제

In [ ]:
# 데이터 준비
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.rc('font', family="NanumBarunGothic") # 한글 폰트 설정

# 타이타닉 데이터 불러오기
df = sns.load_dataset("titanic")

df.head()

### Step 1. 데이터 탐색
> 분석을 시작하기 전에 데이터가 어떻게 생겼는지, 어떤 열이 있는지 전체적으로 확인

In [ ]:
# df.shape : 데이터 프레임의 (행의 수, 열의 수)를 가져옴
print(f"데이터의 크기 : {df.shape}")

In [ ]:
# df.info() : 전체적인 데이터 확인
df.info()

In [ ]:
# df.describe() : 수치형(숫자) 열의 기초 통계 요약
df.describe()

### Step 2. 누락 데이터 처리
> 분석 전에 누락 데이터를 확인 후 처리해야 한다

In [ ]:
# isnull().sum() : 각 열의 누락 데이터 개수 확인
print(df.isnull().sum())

In [ ]:
# 누락 데이터 처리
# 1) age : 177개 누락 => 중앙값으로 대체 
# 2) embarked(항구) : 2개 누락 => 최빈값으로 대체
# 3) deck(선실) : 688개 누락 => 분석에 쓸 수 없으므로 열 자체 삭제

# 1) age : 중앙값으로 대체
age_median = df['age'].median() # 중앙값 계산
print(f"나이 중앙값 : {age_median}")

df['age'] = df['age'].fillna(age_median) # 빈 값을 중앙값으로 채움

In [ ]:
# 2) embarked : 최빈값으로 대체
em_mode = df['embarked'].mode()[0] # 최빈값 계산
print(f"탑승 항구 최빈값 : {em_mode}")

df['embarked'] = df['embarked'].fillna(em_mode)
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])

In [ ]:
# 3) deck : 누락이 너무 많아 열 자체 삭제
df = df.drop(columns=['deck'])

# 처리 후 누락 데이터 확인
print(df.isnull().sum())

### Step 3. 데이터 가공 - 함수 매핑
> 숫자나 영문으로 된 값을 한글로 바꿔 결과를더 직관적으로 만든다.

In [ ]:
# map() + 딕셔너리 : 특정 값을 지정한 값으로 일대일 변환
# - 변환 결과를 기존 열에 덮어쓰지 않고, 새로운 열로 추가하여 원본을 보존

# 1) class -> 객실등급(영어 -> 한글 텍스트)
df['객실등급'] = df['class'].map({'First' : '1등석', 'Second' : '2등석', 'Third' : '3등석'})

# 2) sex -> 성별(영문 -> 한글)
df['성별'] = df['sex'].map({'male' : '남성', 'female' : '여성'})

# 3) survived -> 생존여부(0/1 숫자 -> 의미 있는 텍스트)
df['생존여부'] = df['survived'].map({0 : '사망', 1 : '생존'})

# 변환 결과 확인
df[['class', '객실등급', 'sex', '성별', 'survived', '생존여부']].head()

### Step 4. 연령대 분류 - 구간 분할
> 나이를 그대로 분석하면 너무 세세하기 때문에, 연령대로 묶어서 패턴을 찾아본다.

In [ ]:
# pd.cut() : 지정한 경계값으로 수치 데이터를 구간으로 나눔

# bins : 경계값 리스트
# labels : 각 구간에 붙일 이름
bins = [0, 12, 18, 59, 100]
labels = ["어린이", "청소년", "성인", "노년"]

df['연령대'] = pd.cut(df['age'], bins=bins, labels=labels)

# 연령대별 승객 수 확인
print(df['연령대'].value_counts())

### Step 5. 조건별 탐색 - 필터링
> 특정 조건의 승객만 골라내서 그 그룹의 특징을 집중적으로 분석

In [ ]:
# 불린 인덱싱 : 조건이 True인 행만 선택

# 어린이 승객만 필터링
df_child = df[df['연령대'] == '어린이']
print(f"어린이 승객 수 : {len(df_child)}")

# 어린이 승객의 생존률 확인
print(f"어린이 생존률 : {df_child['survived'].mean() : .2f}")

In [ ]:
# 여러 조건을 &(and), |(or)으로 연결하여 복합 조건으로 필터링
# 1등석이면서 여성인 승객만 필터링 

df_female = df[(df['객실등급'] == '1등석') & (df['성별'] == '여성')]
print(f"1등석 여성 승객 수 : {len(df_female)}명")
print(f"1등석 여성 생존률 : {df_female['survived'].mean() : .2f}")

In [ ]:
# isin() : 여러 값 중 하나에 해당하는 행만 필터링
# 1등석 또는 2등석 승객만 필터링
df_upper = df[df['객실등급'].isin(['1등석', '2등석'])]

print("1·2등석 승객 수:", len(df_upper))
print("1·2등석 생존율:", round(df_upper['survived'].mean(), 2))

### Step 6. 그룹 통계 - 그룹 연산
> 각 그룹별로 통계를 계산하여 "어떤 그룹이 생존률이 높은가"를 확인

In [ ]:
# groupby('연령대') : 연령대 값(어린이/청소년/성인/노년)을 기준으로 데이터를 그룹화
# ['survived'].mean() : 각 그룹의 생존율(평균) 계산
survived_by_age = df.groupby('연령대')['survived'].mean()

round(survived_by_age, 2)

In [ ]:
# 객실등급별로 생존율, 평균 나이, 평균 요금을 한 번에 계산
stats_by_class = df.groupby('객실등급')[['survived', 'age', 'fare']].mean()

# 열 이름을 보기 좋게 정리
stats_by_class.columns = ['평균생존율', '평균나이', '평균요금']

round(stats_by_class, 2)

### Step 7. 교차 분석 - 피벗 테이블
> 두가지 기준을 동시에 적용하여 생존율을 표 형태로 비교

In [ ]:
# pivot_table() : 두 기준(행·열)을 동시에 사용하여 교차 집계표를 만드는 함수
# values='survived' : 집계할 열 → 생존여부
# index='객실등급'   : 표의 행(세로축) 기준
# columns='성별'    : 표의 열(가로축) 기준
# aggfunc='mean'    : 집계 방식 → 평균(생존율)
pivot_result = pd.pivot_table(
    df,
    values='survived',
    index='객실등급',
    columns='성별',
    aggfunc='mean'
)

round(pivot_result, 2)

In [ ]:
# crosstab() : 두 기준의 조합별 건수(승객 수)를 집계
# 객실등급별 / 생존여부별 승객 수 확인
ct_result = pd.crosstab(df['객실등급'], df['생존여부'])

ct_result

In [ ]:
# normalize='index' : 각 행(객실등급)을 기준으로 비율 변환
# 등급별로 사망/생존 비율을 쉽게 비교할 수 있다
ct_ratio = pd.crosstab(df['객실등급'], df['생존여부'], normalize='index')

round(ct_ratio, 2)

**Step 8. 시각화 - 분석 결과를 그래프로 표현**

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', family='NanumBarunGothic')  # 한글 폰트 설정

# Step 6에서 계산한 객실등급별 평균 생존율 데이터 준비
survived_by_class = df.groupby('객실등급')['survived'].mean()

plt.figure(figsize=(7, 5))

# bar() : 막대 그래프
# 객실등급별 생존율을 막대의 높이로 표현 → 등급 간 비교가 직관적
plt.bar(
    survived_by_class.index,    # x축 : 객실등급 이름
    survived_by_class.values,   # y축 : 평균 생존율
    color=['steelblue', 'mediumseagreen', 'salmon']  # 등급마다 다른 색
)

plt.title("객실 등급별 평균 생존율", fontsize=15)
plt.xlabel("객실 등급", fontsize=12)
plt.ylabel("평균 생존율", fontsize=12)
plt.ylim(0, 1)      # y축 범위를 0~1로 고정 (생존율은 0~1 사이)
plt.grid(axis='y')  # y축 방향으로만 격자 표시
plt.show()

In [ ]:
plt.rc('font', family='NanumBarunGothic')  # 한글 폰트 설정

# 생존자(survived=1)와 사망자(survived=0)의 나이 데이터를 각각 분리
age_survived = df[df['survived'] == 1]['age']   # 생존자 나이
age_dead     = df[df['survived'] == 0]['age']   # 사망자 나이

plt.figure(figsize=(9, 5))

# hist() : 히스토그램 — 데이터가 특정 구간에 얼마나 몰려 있는지 분포를 확인
# alpha=0.7 : 투명도를 주어 두 그래프가 겹쳐도 뒤쪽이 보이도록 설정
plt.hist(age_survived, bins=20, color='steelblue', alpha=0.7, label='생존')
plt.hist(age_dead,     bins=20, color='salmon',    alpha=0.7, label='사망')

plt.title("생존 여부별 나이 분포 비교", fontsize=15)
plt.xlabel("나이", fontsize=12)
plt.ylabel("승객 수 (해당 나이 구간에 속하는 인원)", fontsize=12)
plt.legend(fontsize=11)
plt.grid(axis='y')
plt.show()

In [ ]:
plt.rc('font', family='NanumBarunGothic')  # 한글 폰트 설정

# 생존여부별 승객 수 계산
counts = df['생존여부'].value_counts()   # '생존', '사망' 각각의 건수

plt.figure(figsize=(6, 6))

# pie() : 원형 그래프 — 전체에서 각 항목이 차지하는 비율을 시각화
# autopct="%.1f%%" : 각 조각 안에 소수점 1자리 퍼센트로 표시
# explode          : '생존' 조각을 살짝 떼어내 강조
# startangle=90    : 첫 번째 조각이 12시 방향(90도)에서 시작
plt.pie(
    counts,
    labels=counts.index,
    autopct="%.1f%%",
    colors=['steelblue', 'salmon'],
    explode=[0.05, 0],     # 첫 번째 조각(생존)을 살짝 강조
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}  # 조각 사이 흰 테두리
)

plt.title("전체 생존 / 사망 비율", fontsize=15)
plt.show()

과제

## 문제 1) pivot_table()을 활용한 다차원 집계 (Titanic 데이터)

- Seaborn의 titanic 데이터셋을 로드.

- 행(index)은 'class'(객실 등급), 열(columns)은 'survived'(생존 여부)로 설정하여 승객들의 'age'(나이) 평균을 집계하는 피벗테이블 생성.

- margins=True 옵션을 적용하여 각 등급별 전체 평균과 생존 여부별 전체 평균(All)이 포함되도록 설정.

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

result = pd.pivot_table(df, values='age', index='class', columns='survived', aggfunc='mean', margins=True)

result.round(2)

## 문제 2) crosstab()을 활용한 비율 분석 (Tips 데이터)

- Seaborn의 tips 데이터셋을 로드.

- 요일('day')과 시간대('time')의 조합별 손님 수를 crosstab()으로 집계.

- normalize='index' 옵션을 사용하여 각 요일 내에서 점심(Lunch)과 저녁(Dinner) 손님의 비율을 계산.

- 결과값은 소수점 둘째 자리까지 반올림하여 출력.

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("tips")

result = pd.crosstab(df['day'], df['time'], normalize='index')

result.round(2)

## 문제 3) Tips 데이터를 활용한 고객 유형별 분석

- Seaborn의 tips 데이터셋을 사용하여 다음 단계에 따라 분석을 수행.

1. 데이터 준비 및 확인: tips 데이터셋을 로드하고, 모든 열의 누락 데이터 개수를 확인하세요.

2. 구간 분할(Binning): 식사 인원 수('size')를 기준으로 다음과 같이 구간을 나누어 '인원유형' 열을 추가.

    - 1~2명: '소규모'

    - 3~4명: '중규모'

    - 5~6명: '대규모'

    - (힌트: bins=[0, 2, 4, 6], labels=['소규모', '중규모', '대규모'])

3. 필터링 및 그룹 연산:

    3-1) '대규모' 식사(인원유형 == '대규모') 데이터만 필터링하여 출력.

    3-2) 성별('sex')에 따른 평균 식사 금액('total_bill')을 계산하여 출력.

4. 피벗 테이블: 행은 시간대('time'), 열은 흡연 여부('smoker')로 설정하여 평균 팁('tip')을 집계하는 표를 만드세요.

5. 시각화: 요일('day')별 평균 식사 금액('total_bill')을 막대 그래프로 시각화.

    - 제목: "요일별 평균 식사 금액"

    - 컬러: 자유롭게 지정

    - 격자(grid) 표시 포함

In [ ]:
# 나눔 폰트 설치 (한글이 깨지지 않도록 설정)
!sudo apt-get install -y fonts-nanum    # 나눔 폰트 패키지 설치
!sudo fc-cache -fv                      # 폰트 캐시(폰트 목록) 갱신
!rm ~/.cache/matplotlib -rf             # matplotlib 폰트 캐시 삭제

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("tips")

print(df.isnull().sum())

In [ ]:
bins=[0, 2, 4, 6]
labels=['소규모', '중규모', '대규모']

df['인원유형'] = pd.cut(df['size'], bins=bins, labels=labels)
df[['size', '인원유형']].head()

In [ ]:
df_big = df[df['인원유형'] == '대규모']
df_big

In [ ]:
df_sex = df.groupby('sex')['total_bill'].mean()
df_sex.round(2)

In [ ]:
result_pivot = pd.pivot_table(
    df,
    index='time',
    columns='smoker',
    values='tip'
)
result_pivot.round(2)

In [ ]:
import matplotlib.pyplot as plt
plt.rc('font', family='NanumBarunGothic')

result = df.groupby('day')['total_bill'].mean()

plt.figure(figsize=(7, 5))

plt.bar(
    result.index,
    result,
    color=['steelblue', 'mediumseagreen', 'salmon', 'green']
)
plt.title('요일별 평균 식사 금액')
plt.grid(axis='y', linestyle='--')
plt.xlabel('요일')
plt.ylabel('평균금액($)')
plt.show()